In [1]:
import os
from tqdm.auto import tqdm
import pandas as pd

# to use data structures
from pydantic import BaseModel, ConfigDict
# from enum import Enum

# OpenAI API
from openai import OpenAI

# Load the API keys from .env
from dotenv import load_dotenv

pd.options.mode.chained_assignment = (
    None  # default='warn' This disables warning of "copying a slice of a DataFrame"
)
tqdm.pandas()  # activate the tqdm for pandas

In [2]:
# load the API keys
load_dotenv(".env")
for api_key in ["OPENAI_API_KEY"]:
    if os.getenv(api_key) is not None:
        print(api_key, "loaded")
    else:
        print(api_key, "missing")
        print("Please create a .env file with the corresponding API key")

OPENAI_API_KEY loaded


In [3]:
from utils.set_paths import (
    path_app_context,
    path_templates,
    path_data,
    path_output,
)

path_batch_files = path_data / "batches"

# import util functions
from utils.content_utils import *
from utils.file_ops import *

In [5]:
summary_excel = pd.read_excel("../EconJM_status.xlsx", sheet_name="sum")
summary_excel.tail(10)

,add_id,good_match,app_status,institution,location,add_rank,add_field,add_category,deadline_target,review_deadline,...,add_how_apply,other_link,how_to_rec_letters,need_cover_letter,need_RS,need_TS,need_diversity,OMER_letter_status,KLAUS_letter_status,WOO_letter_status
41,u_chile_ig,1.0,generating docs,Universidad de Chile,NaN,assistant,NaN,research,2025-10-17,2025-12-01,...,NaN,NaN,NaN,1,0,0,0,NaN,NaN,NaN
42,uandes_col,1.0,generating docs,"Universidad de los Andes, Colombia",NaN,assistant,NaN,research,2025-11-01,2025-11-14,...,NaN,NaN,NaN,1,0,0,0,NaN,NaN,NaN
43,fashion,1.0,late app,Fashion Institute of Technology,NaN,assistant,NaN,research,2025-09-02,2025-09-02,...,https://fitnyc.interviewexchange.com/jobofferd...,NaN,NaN,1,0,1,0,NaN,NaN,NaN
44,cerdi,1.0,generating docs,CERDI (Centre d'Etude et de Recherches sur le ...,NaN,postdoc,NaN,research,2025-11-20,2025-11-30,...,NaN,NaN,NaN,1,1,0,0,NaN,NaN,NaN
45,crese,1.0,generating docs,Centre de Recherche sur les Stratégies Economi...,NaN,postdoc,NaN,research,2025-11-20,2025-12-01,...,NaN,NaN,NaN,1,1,0,0,NaN,NaN,NaN
46,uab,NaN,generating docs,Universitat Autònoma de Barcelona,NaN,assistant,NaN,research,2025-10-20,2025-11-02,...,NaN,NaN,NaN,1,0,0,0,NaN,NaN,NaN
47,cornell,NaN,generating docs,Cornell University,NaN,assistant,economic history,research,2025-11-05,2025-11-10,...,https://academicjobsonline.org/ajo/jobs/30626.,NaN,NaN,1,1,1,0,NaN,NaN,NaN
48,ucl,1.0,generating docs,University College London,NaN,assistant,NaN,research,2025-11-15,2025-11-21,...,NaN,NaN,NaN,1,0,0,0,NaN,NaN,NaN
49,uc_davis,NaN,generating docs,UC Davis,NaN,assistant,agriculture,research,2025-10-20,2025-11-03,...,https://apptrkr.com/6572257,NaN,NaN,1,1,1,0,NaN,NaN,NaN
50,vassar,1.0,generating docs,Vassar College,NaN,assistant,NaN,research,2025-10-20,2025-11-03,...,NaN,NaN,NaN,1,1,1,0,NaN,NaN,NaN


# Load the needed input files

- Research Statement Template
- Cover Letter Template
- Teaching Statement Template

In [6]:
# Research Statement template
with open(path_templates / "base_text/research_statement.txt", "r") as f:
    lines = f.readlines()

RA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/teaching_statement.txt", "r") as f:
    lines = f.readlines()

TA_TEMPLATE = " ".join(lines)

with open(path_templates / "base_text/cover_letter.txt", "r") as f:
    lines = f.readlines()

CL_TEMPLATE = " ".join(lines)

# Figure out what docs are needed to generate

In [7]:
columns_to_show = [
    "add_id",
    "institution",
    "location",
    "add_rank",
    "add_field",
    "add_category",
    "need_cover_letter",
    "need_RS",
    "need_TS",
    "need_diversity",
]

In [8]:
completed_docs = [str(d).split("/")[-1] for d in path_output.iterdir() if d.is_dir()]

docs_to_generate = set(summary_excel.add_id.to_list()) - set(completed_docs)

if docs_to_generate:
    print("Need to generate docs for:")
    to_generate_df = summary_excel[summary_excel.add_id.isin(docs_to_generate)][
        columns_to_show
    ]
    docs_list = to_generate_df.astype(str).values.tolist()
    for id in docs_to_generate:
        print(f"- {id}")
else:
    print("No docs needed to generate")

Need to generate docs for:
- usydney_1
- monash
- uab
- ie
- puc_cl_1
- uandes_col
- cemfi
- queensland
- fashion
- vassar
- crese
- tec
- u_chile_ig
- uc_davis
- ucl
- sciencepo
- pompeu_fabra
- cerdi
- stockholm_1
- itam
- uc3_1
- melbourne
- cornell
- touluse_1


In [9]:
to_generate_df.head()

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS,need_diversity
15,uc3_1,Universidad Carlos III,NaN,assistant,NaN,research,1,0,0,0
16,usydney_1,University of Sydney,NaN,postdoc,culture,research,1,0,0,0
20,touluse_1,Université Toulouse Capitole,NaN,assistant,NaN,research,1,0,0,0
21,stockholm_1,Stockholm University,NaN,assistant,NaN,research,1,0,0,0
28,puc_cl_1,Pontificia Universidad Catolica de Chile,NaN,assistant,NaN,research,1,0,0,0


# Add the context and additional data for each prompt

For these all the `raw` documents are transcribed in a few bullet points:

- For Job add use 
    ```
    summarize the job add in a few bullet points. Prioritize the attributes, qualitites they look for in a candidate
    ```
- For the isntitutional values use 
    ```
    summarize the institution mission and values in a few bullet points
    ```
- For econ research department use 
    ```
    summarize the economics department or center values and research topics in a few bullet points. Make emphasis on potential co-authors and researcg/teaching groups described there and write it in the file
    ```

In [10]:
# get the add description context
to_generate_df.loc[:, "add_description"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START JOB DESCRIPTION",
        end_marker="END JOB DESCRIPTION",
    ),
    axis=1,
)

to_generate_df.loc[:, "econ_context"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START ECON DEPT",
        end_marker="END ECON DEPT",
    ),
    axis=1,
)

to_generate_df.loc[:, "institution_values"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START INSTITUTIONAL DESCRIPTION",
        end_marker="END INSTITUTIONAL DESCRIPTION",
    ),
    axis=1,
)

# OpenAI Batches

In [11]:
openai_client = OpenAI()
model_name = "gpt-4.1-mini"

In [31]:
# Load the OpenAI batching df
# load history of batches
if (path_batch_files / "batches_history.csv").exists():
    batch_history = pd.read_csv(path_batch_files / "batches_history.csv")
else:
    print("No history of batches found")
    batch_history = pd.DataFrame(
        columns=[
            "created_at",
            "description",
            "model",
            "cat_type",
            "batch_id",
            "status",
            "processing_status",
            "input_file_id",
            "output_file_id",
            "error_file_id",
        ]
    )

batch_history = update_batch_status(batch_history, openai_client)
batch_history.sort_values(["created_at", "model", "cat_type"], inplace=True)
batch_history.to_csv(path_batch_files / "batches_history.csv", index=False)


# Check the inprocess batches
in_process_batches = batch_history[batch_history["status"] == "in_progress"]
print("In-process batches:")
display(in_process_batches.tail(5))

# Check the completed batches
completed_batches = batch_history[
    (batch_history["status"] == "completed")
    & (batch_history["processing_status"] == False)
    & (batch_history["error_file_id"].isna())
]
print("Completed batches:")
display(completed_batches.tail(5))

In-process batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id
24,2025-10-16 22:07:47.964447,Generate Cover Letters,gpt-4.1-mini,cl,batch_68f1b3049c7881908820ec752e375cbf,in_progress,False,file-KanzowLrstDpdQgBdzHnab,NaN,NaN
25,2025-10-16 22:07:55.484722,Generate Research Statements,gpt-4.1-mini,rs,batch_68f1b30c0448819098910b487e592144,in_progress,False,file-7uZbCo1H1E2Xed4HSH8JPN,NaN,NaN
26,2025-10-16 22:08:02.937945,Generate Teaching Statements,gpt-4.1-mini,ts,batch_68f1b313cda881908159e38fe26a9e66,in_progress,False,file-DLoeYzGnXdBxabYLZt8s9U,NaN,NaN


Completed batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id


In [13]:
# batch_history

In [14]:
class BodyText(BaseModel):
    body_text: str
    cot: str
    model_config = ConfigDict(extra="forbid")

# Cover Letter

In [16]:
gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()
# gen_aux

In [17]:
def cl_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to add a position fit paragraph to the following COVER LETTER START and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    Keep the length of the paragraph to about 100 words.
    Use a professional and academic tone, but make it engaging and easy to read.
    For each statement of fit add an EXAMPLE from my research, teaching, or service experience that demostrates the fit statement.
    For example, for interdisciplinary research you can add that I won Dedman College Interdisciplinary Institute’s Inaugural Graduate Student Summer Research and Writing Fellowship.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full new paragraph as `body_text` and a short chain-of-thought explanation of how you modified the BASE COVER LETTER START to fit the institution and department as `cot`.
    The text should follow the Typst formatting in the BASE COVER LETTER START.

    BASE COVER LETTER START:
    {CL_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [18]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: cl_prompt(row),
    axis=1,
)

In [19]:
name_jsonl = "gen_cl"
description_batch = "Generate Cover Letters"

content_schema = BodyText
cat_type = "cl"

In [20]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and CL
Updating the existing old_docs
Need to generate 6 CL docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_68f1b3049c7881908820ec752e375cbf and updated the batch history.


# Research Statement

In [21]:
gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()
# gen_aux

In [22]:
def rs_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE RESEARCH STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my research fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the research statement to about 600 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full research statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE RESEARCH STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst format.

    BASE RESEARCH STATEMENT:
    {RA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [23]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: rs_prompt(row),
    axis=1,
)

In [24]:
name_jsonl = "gen_rs"
description_batch = "Generate Research Statements"

content_schema = BodyText
cat_type = "rs"

In [25]:
%run -i batched_chat_gpt.py

Need to generate 4 RS docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_68f1b30c0448819098910b487e592144 and updated the batch history.


# Teaching Statement

In [26]:
gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()
# gen_aux

In [27]:
def ts_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE TEACHING STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my educational philosophy fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the teaching statement to about 300 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full teaching statement as `body_text` and a short chain-of-thought explanation of how you modified the BASE TEACHING STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst format.

    BASE TEACHING STATEMENT:
    {TA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [28]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: ts_prompt(row),
    axis=1,
)

In [29]:
name_jsonl = "gen_ts"
description_batch = "Generate Teaching Statements"

content_schema = BodyText
cat_type = "ts"

In [30]:
%run -i batched_chat_gpt.py

Need to generate 3 TS docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_68f1b313cda881908159e38fe26a9e66 and updated the batch history.


# Generate the Docs

Here we assume that all docs have been generated

In [29]:
def generate_docs(df, generated_df, gen_type="rs"):
    if gen_type == "rs":
        dir_name = "research_statement"
        file_name = "Gonzalez_RS.typ"
    elif gen_type == "ts":
        dir_name = "teaching_statement"
        file_name = "Gonzalez_TS.typ"
    elif gen_type == "cl":
        dir_name = "cover_letter"
        file_name = "Gonzalez_CL.typ"
    else:
        raise ValueError("gen_type not recognized")

    for i, row in df.iterrows():
        print(f"Processing {row.add_id}...")

        new_content = generated_df[
            generated_df.add_id == row.add_id
        ].body_text_1.values[0]

        target_path = path_output / row.add_id
        subdir_path = target_path / dir_name

        if not target_path.exists():
            print(f"Creating directory: {target_path}")
            copy_and_rename_dir(path_templates / f"{dir_name}", target_path, dir_name)
        else:
            print(f"Main Directory already exists: {target_path}")
            if not subdir_path.exists():
                print(f"Creating subdirectory: {subdir_path}")
                copy_and_rename_dir(
                    path_templates / f"{dir_name}", target_path, dir_name
                )

        try:
            add_lines_to_typs_doc(subdir_path / file_name, new_content)
            print(f"{file_name} added for {row.add_id}")
        except Exception as e:
            print(f"Error processing {row.add_id}: {e}")

## Cover Letter

In [30]:
cl_gen_df = pd.read_csv(path_output / "cl_docs_gpt-4.1-mini.csv")
cl_gen_df

,add_id,body_text_1,cot_1
0,aist_1,My research aligns closely with the Institute ...,I crafted the paragraph to directly connect th...
1,american_u_1,My research aligns closely with American Unive...,I crafted a position fit paragraph that links ...
2,american_u_2,My research interests align closely with Ameri...,I crafted a paragraph that connects my researc...
3,analysis_1,My research aligns closely with Analysis Group...,I examined Analysis Group's institution values...
4,analysis_2,My interdisciplinary expertise aligns well wit...,I introduced a new paragraph emphasizing align...
5,babson_1,I am particularly drawn to Babson College's co...,To tailor the cover letter to Babson College's...
6,babson_2,My research and teaching philosophy align deep...,I crafted a position fit paragraph that connec...
7,bates_1,My research agenda aligns closely with Bates C...,To integrate the institution's values and depa...
8,macmaster_1,I am particularly drawn to McMaster University...,I crafted a paragraph that explicitly connects...
9,nova_lisboa_1,Building on my expertise in cultural economics...,The original cover letter start focused on spe...


In [31]:
gen_aux = to_generate_df[to_generate_df.need_cover_letter == 1].copy()

generate_docs(gen_aux, cl_gen_df, gen_type="cl")

Processing penn_state_1...
Creating directory: ../output_docs/penn_state_1
Gonzalez_CL.typ added for penn_state_1
Processing uc3_1...


IndexError: index 0 is out of bounds for axis 0 with size 0

## Research Statement

In [ ]:
rs_gen_df = pd.read_csv(path_output / "rs_docs_gpt-4.1-mini.csv")
rs_gen_df

,add_id,body_text_1,cot_1
0,babson_2,Research in entertainment media reveals it as ...,I revised the original research statement to c...
1,bates_1,"Entertainment media such as television shows, ...",To tailor the original research statement for ...
2,macmaster_1,"Entertainment media, encompassing television s...",To tailor the base research statement to McMas...
3,simon_frazer_1,"Research Statement\n\nEntertainment media, inc...",I started with the base research statement emp...
4,tamu_1,"Entertainment media such as television shows, ...",I modified the base research statement by expl...
5,waseda_1,# Research Statement\n\nEntertainment media su...,To tailor the base research statement for Wase...
6,pompeu_fabra,# Research Statement\n\nEntertainment media in...,To adapt the base research statement for the U...
7,tec,"# Research Statement\n\nEntertainment media, i...",To tailor the BASE RESEARCH STATEMENT for Tecn...


In [ ]:
gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()

generate_docs(gen_aux, rs_gen_df, gen_type="rs")

Processing pompeu_fabra...
Main Directory already exists: ../output_docs/pompeu_fabra
Creating subdirectory: ../output_docs/pompeu_fabra/research_statement
Gonzalez_RS.typ added for pompeu_fabra
Processing tec...
Main Directory already exists: ../output_docs/tec
Creating subdirectory: ../output_docs/tec/research_statement
Gonzalez_RS.typ added for tec


## Teaching Statement

In [ ]:
rt_gen_df = pd.read_csv(path_output / "ts_docs_gpt-4.1-mini.csv")
rt_gen_df

,add_id,body_text_1,cot_1
0,american_u_1,My goal as an educator aligns deeply with Amer...,I tailored the base teaching statement to refl...
1,american_u_2,My goal as an educator is to make the powerful...,"I integrated American University’s mission, vi..."
2,babson_1,My teaching philosophy centers on making econo...,I revised the base teaching statement by empha...
3,babson_2,# Teaching Statement\n\nMy pedagogical aim is ...,I integrated the institution's mission by emph...
4,bates_1,My goal as an educator is to make the powerful...,To tailor the base teaching statement for Bate...
5,macmaster_1,My overarching goal as an educator is to make ...,I integrated key elements from McMaster's inst...
6,simon_frazer_1,My teaching philosophy at Simon Fraser Univers...,I integrated SFU's institutional values by emp...
7,tamu_1,My goal as an educator is to make the powerful...,I integrated Texas A&M University's institutio...
8,waseda_1,# Teaching Statement\n\nMy goal as an educator...,I enhanced the original teaching statement by ...
9,will_college_1,My goal as an educator is to make the powerful...,I integrated elements from Williams College's ...


In [ ]:
gen_aux = to_generate_df[to_generate_df.need_TS == 1].copy()

generate_docs(gen_aux, rt_gen_df, gen_type="ts")

Processing pompeu_fabra...
Main Directory already exists: ../output_docs/pompeu_fabra
Creating subdirectory: ../output_docs/pompeu_fabra/teaching_statement
Gonzalez_TS.typ added for pompeu_fabra
Processing ie...
Main Directory already exists: ../output_docs/ie
Creating subdirectory: ../output_docs/ie/teaching_statement
Gonzalez_TS.typ added for ie
Processing tec...
Main Directory already exists: ../output_docs/tec
Creating subdirectory: ../output_docs/tec/teaching_statement
Gonzalez_TS.typ added for tec
